In [ ]:
import os, gc, logging
os.environ["NUMBA_NUM_THREADS"] = str(os.cpu_count())
os.environ["NUMBA_THREADING_LAYER"] = "tbb"

from pathlib import Path

import numpy as np
import polars as pl

import plotly.io as pio
import plotly.express as px

import torch
from sentence_transformers import SentenceTransformer

from pacmap import PaCMAP
from hdbscan import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

In [ ]:
# Configure data, models, device, seeds for reproducibility, and plot configurations

TRAIN_PATH = "../../data/train.csv"

OUTPUT_CSV = "../../plots/train_cluster_exemplars.csv"
OUTPUT_PLOT = "../../plots/2d_cluster.png"

OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL_COL = "answer"

EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
EMBED_DIM = 1024

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42

N_LOWEST = 1 # samples closest to each cluster centroid
N_HIGHEST = 1 # samples furthest from each cluster centroid

COL1 = '#00040A'
COL2 = '#202124'
COL3 = '#E1E1E0'
GRID = '#525458'

TITLE_FONT_SIZE = 20
LABEL_FONT_SIZE = 16
TICK_FONT_SIZE = 12

PLOT_WDTH = 1150
PLOT_HGHT = 800
pio.renderers.default = 'notebook'

# Configure logging levels to hide model-loading report
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

disable_progress_bar()

In [3]:
# Load data, combining prompt with all options (A-E) into a single text block per row for embedding
train_data_eda = pl.read_csv(TRAIN_PATH)

# Concatenate prompt + all options into one embeddable text block per row
train_data_eda = train_data_eda.with_columns(
    (
        pl.lit("Question: ") + pl.col("prompt")
        + pl.concat_str(
            [pl.lit(f"\n{opt}) ") + pl.col(opt) for opt in OPTION_COLS]
        )
    ).alias("combined_text")
)

ids = train_data_eda["id"].to_list()
prompts = train_data_eda["prompt"].to_list()
combined_texts = train_data_eda["combined_text"].to_list()
labels = train_data_eda[LABEL_COL].to_list()

n_rows = train_data_eda.height
print(f"Loaded {n_rows} rows.")

Loaded 2000 rows.


In [4]:
# Get high-dimensional embeddings from combined prompt + options text
model = SentenceTransformer(EMBED_MODEL, device=DEVICE, model_kwargs={"dtype": torch.float16})

embeddings = model.encode(
    combined_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

assert embeddings.shape == (n_rows, EMBED_DIM), f"Unexpected shape: {embeddings.shape}"

del model
gc.collect()
torch.cuda.empty_cache()

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [5]:
# Reduce high-dimensional embeddings to 10 dimensions using PaCMAP for clustering
pacmap = PaCMAP(
    n_components=10,
    n_neighbors=30,
    apply_pca=True,
    distance="euclidean",
    random_state=SEED
)

pacmap_data = pacmap.fit_transform(embeddings)

Note: `n_components != 2` have not been thoroughly tested.


In [6]:
# Reduce PaCMAP embedding to 2 dimensions for visualization
# of local groupings, and cluster on 10-dimensional PaCMAP embedding

tsne_2d = TSNE(
    n_components=2,
    random_state=SEED
)

tsne_data_2d = tsne_2d.fit_transform(pacmap_data) # type: ignore

clusterer_2d = HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

cluster_labels_2d = clusterer_2d.fit_predict(pacmap_data) # type: ignore

In [7]:
# Construct plotting DataFrame

plot_2d = pl.DataFrame({
    "X": tsne_data_2d[:, 0], # type: ignore
    "Y": tsne_data_2d[:, 1], # type: ignore
    "ID": ids,
    "Cluster": cluster_labels_2d,
    "Prompt": prompts
})

plot_2d = plot_2d.with_columns(
    pl.Series("Cluster", cluster_labels_2d).cast(pl.Utf8)
)

cluster_order_2d = (
    plot_2d
    .select(pl.col("Cluster").cast(pl.Int64).unique().sort(descending=True))
    .to_series()
    .cast(pl.Utf8)
    .to_list()
)

In [ ]:
# 2d_cluster
# Plot the 2d scatter plot using computed data

fig = px.scatter(
    plot_2d,
    x="X",
    y="Y",
    color="Cluster",
    hover_data=["ID", "Cluster", "Prompt"],
    color_discrete_sequence=px.colors.qualitative.Set1,
    category_orders={"Cluster": cluster_order_2d}
)

fig.update_traces(marker=dict(size=4, opacity=0.8))

fig.update_layout(
    width=PLOT_WDTH,
    height=PLOT_HGHT,
    paper_bgcolor=COL1,
    plot_bgcolor=COL1,
    font=dict(color=COL3),
    hovermode="closest",

    title=dict(
        text="2D PaCMAP to t-SNE Projection of Prompt + Options Text clustered with HDBSCAN",
        font=dict(color=COL3, size=TITLE_FONT_SIZE),
        x=0.5, xanchor="center"
    ),

    legend=dict(
        title=dict(text="Cluster", font=dict(color=COL3)),
        font=dict(color=COL3),
        bgcolor=COL1
    ),

    scene=dict(
        bgcolor=COL1,
        xaxis=dict(
            color=COL3,
            gridcolor=GRID,
            gridwidth=1,
            backgroundcolor=COL1
        ),
        yaxis=dict(
            color=COL3,
            gridcolor=GRID,
            gridwidth=1,
            backgroundcolor=COL1
        )
    )
)

fig.write_image(OUTPUT_PLOT)

fig.show()

In [12]:
# Compute each cluster's centroid in PaCMAP-10D space, 
# then find N_LOWEST closest and N_HIGHEST furthest
# samples from each centroid by Euclidean distance

exemplar_rows = []

unique_clusters = sorted(set(cluster_labels_2d))

for cluster_id in unique_clusters:
    cluster_mask = cluster_labels_2d == cluster_id
    cluster_indices = np.where(cluster_mask)[0]
    cluster_points = pacmap_data[cluster_mask] # type: ignore

    centroid = cluster_points.mean(axis=0, keepdims=True)
    distances = pairwise_distances(cluster_points, centroid, metric="euclidean").flatten()

    order = np.argsort(distances)
    n_available = len(order)

    lowest_n = min(N_LOWEST, n_available)
    highest_n = min(N_HIGHEST, n_available)

    closest_local_idx = order[:lowest_n]
    furthest_local_idx = order[-highest_n:] if highest_n > 0 else np.array([], dtype=int)

    for rank, local_idx in enumerate(closest_local_idx):
        global_idx = cluster_indices[local_idx]
        
        exemplar_rows.append({
            "cluster": int(cluster_id),
            "exemplar_type": "closest",
            "rank": rank + 1,
            "distance_to_centroid": float(distances[local_idx]),
            "id": ids[global_idx],
            "prompt": prompts[global_idx],
            "combined_text": combined_texts[global_idx],
            "label": labels[global_idx]
        })

    for rank, local_idx in enumerate(furthest_local_idx):
        global_idx = cluster_indices[local_idx]
        
        exemplar_rows.append({
            "cluster": int(cluster_id),
            "exemplar_type": "furthest",
            "rank": rank + 1,
            "distance_to_centroid": float(distances[local_idx]),
            "id": ids[global_idx],
            "prompt": prompts[global_idx],
            "combined_text": combined_texts[global_idx],
            "label": labels[global_idx]
        })

exemplars_df = pl.DataFrame(exemplar_rows)
exemplars_df.write_csv(OUTPUT_CSV)

print(f"Saved {exemplars_df.height} exemplar rows across {len(unique_clusters)} clusters to {OUTPUT_CSV}")
exemplars_df.head(10)

Saved 364 exemplar rows across 182 clusters to ../plots/train_cluster_exemplars.csv


cluster,exemplar_type,rank,distance_to_centroid,id,prompt,combined_text,label
i64,str,i64,f64,i64,str,str,str
-1,"""closest""",1,0.732374,521,"""Identify the correct statement…","""Question: Identify the correct…","""D"""
-1,"""furthest""",1,4.680617,77,"""Determine the correct option: …","""Question: Determine the correc…","""E"""
0,"""closest""",1,0.005809,268,"""Identify the correct statement…","""Question: Identify the correct…","""B"""
0,"""furthest""",1,0.089189,1654,"""Identify the correct statement…","""Question: Identify the correct…","""B"""
1,"""closest""",1,0.015075,1530,"""Determine the correct option: …","""Question: Determine the correc…","""C"""
1,"""furthest""",1,0.04049,282,"""Select the most accurate optio…","""Question: Select the most accu…","""C"""
2,"""closest""",1,0.032998,1272,"""Determine the correct option: …","""Question: Determine the correc…","""B"""
2,"""furthest""",1,0.112336,1119,"""Pick the best possible answer:…","""Question: Pick the best possib…","""B"""
3,"""closest""",1,0.01405,1157,"""Pick the best possible answer:…","""Question: Pick the best possib…","""C"""


#### Note: The coordinates in the plot come from t-SNE reduced values, while the cluster assignments and centroid-distance calculations are based on 10-dimensional PaCMAP embeddings. PaCMAP is used for clustering to preserve both local and global structure, and t-SNE is used only for visualizing local groupings. Distances between clusters in the t-SNE plot should not be over-interpreted. Centroid distances used to select exemplar rows are computed in the PaCMAP-10D space, not the t-SNE-2D space.